# Napari animation

Loads the raw image and a saved analysis bundle (`{stem}-analysis`) and builds napari layers for animation.


In [ ]:
import json
import os
from pathlib import Path
from typing import Any

import napari
import numpy as np

from napari_animation import Animation
from vistiq.io import ImageLoader, ImageLoaderConfig, unstack_image
from vistiq.graph import load_analysis
from vistiq.graph.napari import add_hierarchical_napari_layers


In [ ]:
# Set image path + scene index used to produce the saved analysis bundle
path = "../data/Animal 1.lif"
scene_index = 0
bundle_dir = Path(path).parent / f"{Path(path).stem}-analysis"
bundle_dir


In [ ]:
ilc = ImageLoaderConfig(
    squeeze=True,
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"},
    scene_index=scene_index,
    split_channels=False,
)
img, metadata = ImageLoader(ilc).run(path)

# Backfill legacy bundles that predate meta.json
meta_path = bundle_dir / "meta.json"
if not meta_path.exists():
    spatial_dir = bundle_dir / "spatial"
    keys = sorted(
        p.name.removesuffix(".distance.parquet")
        for p in spatial_dir.glob("*.distance.parquet")
    )
    origins = sorted({k.split("@", 1)[1] for k in keys if "@" in k})
    meta_path.write_text(
        json.dumps(
            {
                "spatial_origins": origins,
                "spatial_result_keys": keys,
            },
            indent=2,
        )
    )
    print(f"Wrote missing meta.json for legacy bundle: {meta_path}")

bundle = load_analysis(bundle_dir)

measurements = {
    "containment_graph": bundle["containment_graph"],
    "region_analyzer_all": bundle["region_analyzer_all"],
    "spatial_summary": bundle["spatial_summary"],
}
features = bundle["features"]
spatial_results = bundle["spatial_results"]
knn_spatial_cfg = bundle["knn_cfg"]
rnn_spatial_cfg = bundle["rnn_cfg"]
bundle["meta"]


In [ ]:
def add_hierarchical_napari_layers_with_rotate(
    viewer: Any,
    *args,
    rotate: float = 0.0,
    **kwargs,
) -> dict[str, Any]:
    """Notebook wrapper: apply a shared rotate angle to every layer."""
    layers = add_hierarchical_napari_layers(viewer, *args, **kwargs)
    for layer in layers.values():
        layer.rotate = rotate
    return layers


In [ ]:
viewer = napari.Viewer()
rotate = -118.0
scale = metadata["physical_pixel_sizes"]
channel_colors = ("gray", "green", "magenta")

# Add raw channel images for context
ch_images, _ = unstack_image(img, metadata=metadata, axis="C", strict=False)
for name, c_img, color in zip(metadata["channel_names"], ch_images, channel_colors):
    viewer.add_image(
        c_img,
        name=f"{name}",
        rotate=rotate,
        scale=scale,
        colormap=color,
        blending="additive",
        visible=False,
    )

dag = measurements["containment_graph"]
napari_layers = add_hierarchical_napari_layers_with_rotate(
    viewer,
    dag,
    features,
    spatial_results,
    rnn_radius=rnn_spatial_cfg.radius if rnn_spatial_cfg is not None else 15,
    k=knn_spatial_cfg.k if knn_spatial_cfg is not None else 5,
    partner_size=2.0,
    rotate=rotate,
)
list(napari_layers)


# Animate

In [ ]:
from napari_animation import Animation

animation = Animation(viewer)
viewer.update_console({"animation": animation})

In [ ]:
for l in viewer.layers:
    l.visible=True
    if isinstance(l, napari.layers.Image):
        l.opacity=1.0
    else:
        l.opacity=0.0
viewer.dims.ndisplay=2
viewer.camera.angles=(0,0,90)
viewer.dims.current_step=(21,0,0)
#viewer.camera.zoom=5.0

In [ ]:
#membrane_l.opacity=1.0
#membrane_l.gamma=1.1
#nb_l.opacity=1.0
#nb_l.gamma=1.1
#nb_l.contrast_limits=0.75*np.array(nb_l.contrast_limits)
#lines_l.opacity=0.0
#points_l.opacity=0.0
#points_l.out_of_slice_display=True

In [ ]:
animation.capture_keyframe()

viewer.dims.current_step=(5,0,0)
animation.capture_keyframe(steps=24)

viewer.dims.current_step=(85,0,0)
animation.capture_keyframe(steps=120)

viewer.dims.current_step=(45,0,0)
animation.capture_keyframe(steps=60)

for l in viewer.layers:
    if isinstance(l, napari.layers.Image):
        l.opacity=0.0
animation.capture_keyframe(steps=20)

viewer.dims.ndisplay=3
animation.capture_keyframe(steps=5)

viewer.layers["Dpn (homotypic RNN Dpn count)"].opacity=0.95
viewer.layers["Lobe center → Dpn"].opacity=0.1
animation.capture_keyframe(steps=20)

viewer.camera.angles=(0,0,55)
animation.capture_keyframe(steps=90)


#viewer.camera.angles=(0,0,55)
viewer.layers["Dpn (homotypic RNN Dpn count)"].opacity=0.0
viewer.layers["Dpn (EdU+/-)"].opacity=0.95
animation.capture_keyframe(steps=20)

viewer.camera.angles=(0,135,55)
animation.capture_keyframe(steps=90)


#viewer.camera.angles=(0,135,55)
viewer.layers["Dpn (EdU+/-)"].opacity=0.0
viewer.layers["Dpn (heterotypic EdU RNN count)"].opacity=0.95
animation.capture_keyframe(steps=20)

viewer.camera.angles=(0,135,240)
animation.capture_keyframe(steps=90)

#viewer.camera.angles=(0,135,240) #160
viewer.layers["Dpn (heterotypic EdU RNN count)"].opacity=0.0
viewer.layers["Dpn → EdU (heterotypic RNN)"].opacity=0.1
animation.capture_keyframe(steps=20)

viewer.camera.angles=(0,315,240)
animation.capture_keyframe(steps=90)

viewer.camera.angles=(0,355,120)
animation.capture_keyframe(steps=90)

"""

viewer.camera.angles=(0, -30, 165 ) #viewer.camera.angles=(0,0,50)
animation.capture_keyframe(steps=50) #55

viewer.camera.angles=(0, -195, 120) #(0,135,50)
animation.capture_keyframe(steps=50) #180

viewer.camera.angles=(0,-45,195)
viewer.layers["Dpn (homotypic RNN Dpn count)"].opacity=0.0
viewer.layers["Lobe center → Dpn"].opacity=0.1
viewer.layers["Dpn (EdU+/-)"].opacity=0.95
animation.capture_keyframe(steps=50)

viewer.camera.angles=(0,-130,0)
viewer.layers["Dpn (EdU+/-)"].opacity=0.0
viewer.layers["Dpn (heterotypic EdU RNN count)"].opacity=0.95
animation.capture_keyframe(steps=180)

viewer.camera.angles=(0,0,0)
viewer.layers["Dpn (heterotypic EdU RNN count)"].opacity=0.0
viewer.layers["Dpn → EdU (heterotypic RNN)"].opacity=0.1
animation.capture_keyframe(steps=180)


#viewer.camera.angles=(0,135,55)
animation.capture_keyframe(steps=150)

#animation.capture_keyframe(steps=25)
viewer.camera.angles=(0,135,180)

animation.capture_keyframe(steps=250)

#animation.capture_keyframe()
"""

In [ ]:
fname = f"{Path(path).stem}-scene-{scene_index}.mp4"
outdir = Path(path).parent
fpath = os.path.join(outdir, fname)
print (fpath)

animation.animate(fpath, canvas_only=True)